# Freeze the MVP case-ID split manifest — mvp_split_v1

Freeze deterministic chronological membership with seed **20261016**, carried
forward from the source package. Dates determine membership; the seed is recorded
for reproducibility and does not shuffle cases or override chronology.

Training: 2010–2020. Development: 2021–2022. Testing: 2023–2024.
Families spanning period boundaries are excluded. Recorded calibration exposure
propagates across family members; any calibration family outside training is
excluded rather than reassigned. Missing companion identities/dates remain on hold.

This freezes **113 assignments and the pending disposition of Sodhi** within the
114-case selection. It also snapshots provisional labels, without changing their
review status. Human-reviewed targets and family sign-off remain pending. No models
are trained. Changes to cases, labels, families, or assignments require a new version
and rerunning affected experiments; existing versioned files cannot be overwritten.

Calibration coverage uses the shared manifest, saved independent calibration notes,
and saved calibration submission IDs. These identify the same 12 calibration cases.
No additional previously discussed calibration IDs appear in the available records;
any additional exposure supplied later must trigger a version update.

Run all cells from MVP or any directory inside the repository. Python standard
library only. Outputs: `data/mvp_split_v1/split_manifest.json` and its SHA-256 file.


In [1]:
from pathlib import Path
from collections import defaultdict, Counter
from datetime import date
import hashlib, json, re

VERSION = "mvp_split_v1"
SEED = 20261016
FREEZE_DATE = "2026-10-23"
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "annotation/source/cases.jsonl").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside legal-judgment.")
MVP = ROOT / "annotation/submissions/becca/MVP"
OUTPUT = MVP / "data" / VERSION
INPUTS = {
    "metadata": MVP / "data/metadata_v2_family_review/metadata.jsonl",
    "families": MVP / "data/family_v1/case_family_manifest.jsonl",
    "family_provenance": MVP / "data/family_v1/provenance.json",
    "selected_ids": MVP / "data/unanimous_include_v1/case_ids.json",
    "calibration_manifest": ROOT / "annotation/calibration/manifest.json",
    "source_assignments": ROOT / "annotation/source/assignments.json",
}
for reviewer in ("becca", "hyunsang", "yinkui"):
    for filename in ("independent_calibration_notes.md", "calibration.jsonl"):
        p = ROOT / "annotation/submissions" / reviewer / filename
        if p.is_file(): INPUTS[f"{reviewer}_{filename}"] = p

def require(ok, message):
    if not ok: raise ValueError(message)

def read_json(p):
    return json.loads(p.read_text(encoding="utf-8"))

def read_rows(p):
    return [json.loads(s) for s in p.read_text(encoding="utf-8").splitlines() if s.strip()]

def sha(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

def period(value):
    year = date.fromisoformat(value).year
    if 2010 <= year <= 2020: return "train"
    if 2021 <= year <= 2022: return "development"
    if 2023 <= year <= 2024: return "test"
    return "outside_range"

metadata = read_rows(INPUTS["metadata"])
family_cases = read_rows(INPUTS["families"])
selected_ids = read_json(INPUTS["selected_ids"])
provenance = read_json(INPUTS["family_provenance"])
require(sha(INPUTS["metadata"]) == provenance["metadata_v2_family_review/metadata.jsonl"], "Metadata changed")
require(sha(INPUTS["families"]) == provenance["outputs"]["case_family_manifest.jsonl"], "Family manifest changed")
require(read_json(INPUTS["source_assignments"])["seed"] == SEED, "Source seed differs")
require(len(metadata) == len({r["case_id"] for r in metadata}) == 114, "MVP ID coverage")
require(len(selected_ids) == len(set(selected_ids)) == 114 and set(selected_ids) == {r["case_id"] for r in metadata}, "Selected IDs mismatch")
require(len(family_cases) == len({r["case_id"] for r in family_cases}) == 506, "Source family coverage")
lookup = {r["case_id"]: r for r in family_cases}
groups = defaultdict(list)
for r in family_cases: groups[r["case_family_id"]].append(r)

calibration_evidence = defaultdict(set)
for cid in read_json(INPUTS["calibration_manifest"])["case_ids"]:
    calibration_evidence[cid].add(str(INPUTS["calibration_manifest"].relative_to(ROOT)))
for key, p in INPUTS.items():
    if key.endswith("independent_calibration_notes.md"):
        found = set(re.findall(r"NY-\d+", p.read_text(encoding="utf-8")))
    elif key.endswith("calibration.jsonl"):
        found = {r["case_id"] for r in read_rows(p)}
    else:
        continue
    for cid in found: calibration_evidence[cid].add(str(p.relative_to(ROOT)))
require(set(calibration_evidence) <= set(lookup), "Unknown calibration IDs")
calibration_families = {lookup[c]["case_family_id"] for c in calibration_evidence}
calibration_registry = []
for r in family_cases:
    if r["case_family_id"] in calibration_families:
        origins = sorted(m["case_id"] for m in groups[r["case_family_id"]] if m["case_id"] in calibration_evidence)
        calibration_registry.append({
            "case_id": r["case_id"], "case_family_id": r["case_family_id"],
            "training_only": True, "direct_calibration_case": r["case_id"] in calibration_evidence,
            "calibration_origin_case_ids": origins,
            "evidence_paths": sorted({p for c in origins for p in calibration_evidence[c]}),
            "selected_for_mvp": r["case_id"] in set(selected_ids),
        })
print("Direct calibration cases:", len(calibration_evidence))
print("Training-only source records after family propagation:", len(calibration_registry))


Direct calibration cases: 12
Training-only source records after family propagation: 13


In [2]:
records = []
for r in sorted(metadata, key=lambda x: x["case_id"]):
    cid = r["case_id"]; f = lookup[cid]; members = groups[f["case_family_id"]]
    require(r["case_family_id"] == f["case_family_id"] and r["decision_date"] == f["date_filed"], f"Metadata mismatch: {cid}")
    require(r["text_sha256"] == f["text_sha256"], f"Source mismatch: {cid}")
    require(sorted(m["case_id"] for m in members) == sorted(f["family_member_case_ids"]), "Incomplete family")
    observed = {period(m["date_filed"]) for m in members}
    training_only = f["case_family_id"] in calibration_families
    pending = any(m["external_companion_unresolved"] for m in members)
    reasons = []
    if len(observed) > 1: reasons.append("family_spans_period_boundaries")
    if "outside_range" in observed: reasons.append("family_member_outside_2010_2024")
    if training_only and observed != {"train"}: reasons.append("calibration_family_outside_training_period")
    status = "excluded" if reasons else "pending" if pending else "assigned"
    target = r["provisional_target"]
    require(type(target) is int and target in (0, 1), f"Invalid provisional target: {cid}")
    require(all(r[n + "_target"] == target for n in ("becca", "hyunsang", "yinkui")), f"Target disagreement: {cid}")
    records.append({
        "case_id": cid, "case_family_id": f["case_family_id"],
        "decision_date": r["decision_date"], "date_period": period(r["decision_date"]),
        "split": period(r["decision_date"]) if status == "assigned" else None,
        "status": status, "exclusion_reasons": reasons,
        "pending_reasons": ["external_companion_identity_or_date_unresolved"] if pending else [],
        "training_only": training_only,
        "target": target, "target_status": "provisional_unanimous_reviewer_label",
        "target_review_status": r["target_review_status"],
        "reviewed_target": r["target"], "family_human_review_status": "pending",
        "source_url": r["source_url"], "text_sha256": r["text_sha256"],
        "family_member_case_ids": sorted(m["case_id"] for m in members),
        "observed_family_periods": sorted(observed),
    })
splits = {p: sorted(r["case_id"] for r in records if r["split"] == p) for p in ("train", "development", "test")}
family_splits = defaultdict(set)
for r in records:
    if r["split"]:
        family_splits[r["case_family_id"]].add(r["split"])
        require(not r["training_only"] or r["split"] == "train", "Calibration leakage")
        require(r["observed_family_periods"] == [r["split"]], "Family period mismatch")
        require(not r["pending_reasons"], "Pending case assigned")
require(all(len(v) == 1 for v in family_splits.values()), "Family overlap across splits")
assigned_ids = [c for ids in splits.values() for c in ids]
require(len(assigned_ids) == len(set(assigned_ids)), "Duplicate split IDs")
require(set(assigned_ids) | {r["case_id"] for r in records if r["split"] is None} == set(selected_ids), "Missing audit cases")

counts = []
for p in ("train", "development", "test", "outside_range"):
    dated = [r for r in records if r["date_period"] == p]
    if p == "outside_range" and not dated: continue
    kept = [r for r in dated if r["split"] == p]
    classes = Counter(r["target"] for r in kept)
    class_families = {str(label): len({r["case_family_id"] for r in kept if r["target"] == label}) for label in (0, 1)}
    flags = []
    if min(classes[0], classes[1]) == 0: flags.append("one_or_both_classes_absent")
    if len(kept) < 20: flags.append("fewer_than_20_cases")
    if min(classes[0], classes[1]) < 10: flags.append("fewer_than_10_cases_in_a_class")
    counts.append({"period": p, "all_cases": len(dated), "assigned_cases": len(kept),
                   "families": len({r["case_family_id"] for r in kept}),
                   "class_0": classes[0], "class_1": classes[1],
                   "class_family_counts": class_families,
                   "training_only": sum(r["training_only"] for r in kept),
                   "excluded": sum(r["status"] == "excluded" for r in dated),
                   "pending": sum(r["status"] == "pending" for r in dated),
                   "sample_size_flags": flags})
require(sum(c["all_cases"] for c in counts) == 114, "Count total mismatch")
require(all(c["all_cases"] == c["assigned_cases"] + c["excluded"] + c["pending"] for c in counts), "Count reconciliation failed")
require(all(c["class_0"] + c["class_1"] == c["assigned_cases"] for c in counts), "Class total mismatch")
print("Period        Cases  Families  Class 0  Class 1  Excluded  Pending")
for c in counts:
    print(f"{c['period']:12s} {c['assigned_cases']:5d} {c['families']:9d} {c['class_0']:8d} {c['class_1']:8d} {c['excluded']:9d} {c['pending']:8d}")
print("\nSample-size flags (heuristics, not a power analysis):")
for c in counts: print(c["period"], ":", ", ".join(c["sample_size_flags"]) or "no threshold flag; adequacy not established")
print("\nTraining-only MVP IDs:", [r["case_id"] for r in records if r["training_only"]])
print("Pending IDs:", [r["case_id"] for r in records if r["status"] == "pending"])


Period        Cases  Families  Class 0  Class 1  Excluded  Pending
train           81        81       55       26         0        1
development     16        16        8        8         0        0
test            16        16       11        5         0        0

Sample-size flags (heuristics, not a power analysis):
train : no threshold flag; adequacy not established
development : fewer_than_20_cases, fewer_than_10_cases_in_a_class
test : fewer_than_20_cases, fewer_than_10_cases_in_a_class

Training-only MVP IDs: ['NY-2757329', 'NY-4546774', 'NY-4840329', 'NY-6004283']
Pending IDs: ['NY-4655053']


## Interpretation and frozen output

The development set has only 16 cases (8 per class); testing has 16 cases (11/5).
Both are flagged as too small for strong comparative conclusions and should support
only cautious exploratory comparisons unless a later, documented design addresses
sample size. One changed prediction moves accuracy by 6.25 percentage points in
either set; test class-1 recall changes by 20 percentage points per case. Thresholds
are screening heuristics, not a formal power calculation, and do not justify tuning
or changing the split based on model results.

The manifest includes all 114 IDs, 113 assigned cases, Sodhi's hold, all-source
calibration restrictions, actual provisional class/family counts, input checksums,
policy, seed, and version. `target` is explicitly provisional and must not be
represented as human-gold evaluation data. This assignment freeze does not waive
remaining target or family-review requirements.


In [3]:
manifest = {
    "version": VERSION, "seed": SEED, "frozen_on": FREEZE_DATE,
    "status": "frozen_assignments_with_provisional_targets_and_explicit_pending_cases",
    "seed_role": "Recorded source-package seed; deterministic date/family assignments use no randomization",
    "parent_versions": ["unanimous_include_v1", "metadata_v2_family_review", "family_v1"],
    "policy": {
        "periods": {"train": [2010, 2020], "development": [2021, 2022], "test": [2023, 2024]},
        "date_field": "decision_date copied from source date_filed",
        "family_scope": "all 506 frozen source records",
        "cross_period_families": "exclude from primary comparison",
        "calibration": "training-only across entire family; exclude if outside chronological training period",
        "unresolved_companions": "hold without a split until reviewed in a new version",
        "future_chunks": "inherit parent case family and split; never split a family; multi-case annotation packets are not model units",
        "version_changes": "New version required for changed cases, labels, families, exposure, or assignment; rerun affected experiments",
    },
    "sample_size_policy": {"minimum_cases_flag": 20, "minimum_per_class_flag": 10,
                           "interpretation": "Heuristic warnings only; no statistical power established"},
    "label_status": "provisional; human target/family review remains pending",
    "calibration_coverage": "12 recorded calibration IDs plus family propagation; additional unrecorded prior exposure requires a new version",
    "calibration_registry": calibration_registry,
    "selection_count": len(records), "assigned_count": len(assigned_ids),
    "split_case_ids": splits,
    "pending_case_ids": sorted(r["case_id"] for r in records if r["status"] == "pending"),
    "excluded_case_ids": sorted(r["case_id"] for r in records if r["status"] == "excluded"),
    "counts": counts, "cases": records,
    "inputs": {name: {"path": str(p.relative_to(ROOT)), "sha256": sha(p)} for name, p in INPUTS.items()},
}
content = (json.dumps(manifest, ensure_ascii=False, indent=2) + "\n").encode("utf-8")
manifest_hash = hashlib.sha256(content).hexdigest()
files = {OUTPUT / "split_manifest.json": content,
         OUTPUT / "split_manifest.sha256": (manifest_hash + "  split_manifest.json\n").encode("utf-8")}
for p, data in files.items():
    require(not p.exists() or p.read_bytes() == data, f"Frozen version differs: {p}. Choose a new VERSION.")
OUTPUT.mkdir(parents=True, exist_ok=True)
for p, data in files.items():
    if not p.exists(): p.write_bytes(data)
require(read_json(OUTPUT / "split_manifest.json") == manifest, "Saved manifest verification failed")
require(sha(OUTPUT / "split_manifest.json") == manifest_hash, "Saved checksum verification failed")
print("Frozen:", (OUTPUT / "split_manifest.json").relative_to(MVP))
print("Version:", VERSION, "Seed:", SEED)
print("Assigned:", len(assigned_ids), "Pending:", len(manifest["pending_case_ids"]), "Excluded:", len(manifest["excluded_case_ids"]))
print("SHA-256:", manifest_hash)


Frozen: data/mvp_split_v1/split_manifest.json
Version: mvp_split_v1 Seed: 20261016
Assigned: 113 Pending: 1 Excluded: 0
SHA-256: 8130672b660fed6ec2ddce62925583fdd39e6fcbdf993fcb5fc8b10c8f710a73
